In [6]:
from pulp import *
import pandas as pd

masses = pd.read_csv("../masses_simple.tsv", sep="\t").set_index("base", drop=False)

In [7]:
masses

,base,exact_mass
base,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954


In [10]:
import random
true_sequence = "CGGAUCGAUCUG"
# sample random fragments from true sequence
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l, len(true_sequence))
    return l, r
fragments = [random_fragment() for _ in range(20)]
fragment_masses = [sum(masses.loc[b, "exact_mass"] for b in true_sequence[l:r + 1]) for l, r in fragments]
fragments, fragment_masses

([(8, 12),
  (2, 2),
  (0, 2),
  (5, 12),
  (10, 10),
  (2, 11),
  (1, 4),
  (9, 10),
  (7, 7),
  (10, 10),
  (11, 12),
  (0, 4),
  (8, 10),
  (6, 11),
  (3, 7),
  (11, 12),
  (1, 5),
  (1, 7),
  (7, 8),
  (8, 8)],
 [1014.31627,
  283.09167,
  809.26886,
  1807.59021,
  244.06954,
  2601.84817,
  1077.34963,
  487.15506,
  267.09675,
  244.06954,
  283.09167,
  1320.43515,
  731.2246,
  1564.50469,
  1304.44023,
  283.09167,
  1320.43515,
  1870.62357,
  511.16629,
  244.06954])

In [14]:
prob = LpProblem("RNA sequencing", LpMinimize)
# i = 1,...,n: positions in the sequence
# j = 1,...,m: fragments
# b = 1,...,k: (modified) bases

n = 25
m = len(fragment_masses)
bases = "ACGU"

# y: binary variables indicating base at position i
y = [{b: LpVariable(f"y_{i},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for i in range(n)]
# l_j: left end of fragment j (inclusive)
l = [LpVariable(f"l_{j}", lowBound=0, upBound=n + 1, cat=LpInteger) for j in range(m)]
# r_j: right end of fragment j (inclusive)
r = [LpVariable(f"r_{j}", lowBound=0, upBound=n + 1, cat=LpInteger) for j in range(m)]
# x: indicator variable for fragment j presence at position i
x = [[LpVariable(f"x_{i},{j}", lowBound=0, upBound=1, cat=LpInteger) for j in range(m)] for i in range(n)]
# weight_diff_abs: absolute value of weight_diff
weight_diff_abs = [LpVariable(f"weight_diff_abs_{j}", lowBound=0, cat=LpContinuous) for j in range(m)]
# weight_diff: difference between fragment monoisotopic mass and sum of masses of bases in fragment as estimated in the MILP
weight_diff = [fragment_masses[j] - lpSum([y[i][b] * masses.loc[b, "exact_mass"] * x[i][j] for i in range(n) for b in bases]) for j in range(m)]

# optimization function
prob += lpSum([weight_diff_abs[j] for j in range(m)])

for i in range(n):
    prob += lpSum([y[i][b] for b in bases]) == 1

for j in range(m):
    # fragment must have length 1 at least
    prob += l[j] < r[j]
    # ensure that x[i][j] is 1 between l[j] and r[j] and zero otherwise
    for i in range(n):
        prob += x[i][j] <= (r[j] + 1 - i) # while i<=r[j], x can become 1, otherwise it is 0
        prob += x[i][j] <= (i + 1 - l[j]) # while i>=l[j], x can become 1, otherwise it is 0

# constrain weight_diff_abs to be the absolute value of weight_diff
for j in range(m):
    prob += weight_diff_abs[j] >= weight_diff[j]
    prob += weight_diff_abs[j] >= -weight_diff[j]

prob.solve()


TypeError: 'LpVariable' object is not subscriptable